# Tối Thượng: Kết hợp Mạng Nơ-ron (thay cho CNN) + SVM hoàn toàn tự code bằng NumPy

Đây là đỉnh cao của đồ án Học Máy! Chúng ta sẽ **KHÔNG sử dụng bất kỳ thư viện Học Máy (Machine Learning) nào** có sẵn (không `TensorFlow/Keras`, không `Scikit-learn`). Tất cả đều được tự code 100% dựa trên Đại số tuyến tính và Vi tích phân của NumPy.

**Pipeline như sau:**
1. Xây dựng Mạng Nơ-ron (MLP) đóng vai trò là **Feature Extractor** (Bộ trích xuất đặc trưng ảnh) tương tự chức năng của CNN.
2. Chạy ảnh qua mạng Nơ-ron, lấy đầu ra tại **lớp ẩn (128 chiều)** làm vector đặc trưng.
3. Đưa các vector đặc trưng này vào mô hình **SVM Đa lớp (One-vs-Rest)** cũng hoàn toàn tự code bằng thuật toán Gradient Descent.

### BƯỚC 1: XỬ LÝ DỮ LIỆU

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Đọc dữ liệu
train_df = pd.read_csv('../data/digit-recognizer/train.csv')
test_df = pd.read_csv('../data/digit-recognizer/test.csv')

data = np.array(train_df)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_val = data_dev[0]
X_val = data_dev[1:n] / 255.  # Dữ liệu ảnh: 784 x m_val

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n] / 255. # Dữ liệu ảnh: 784 x m_train

X_test_final = np.array(test_df).T / 255.
print("Tập Huấn Luyện (X):", X_train.shape)
print("Tập Kiểm Chứng (X):", X_val.shape)

### BƯỚC 2: MẠNG NƠ-RON (FEATURE EXTRACTOR)

In [ ]:
# --- CÁC HÀM TOÁN HỌC --- 
def init_params():
    W1 = np.random.randn(128, 784) * np.sqrt(2. / 784)
    b1 = np.zeros((128, 1))
    W2 = np.random.randn(10, 128) * np.sqrt(2. / 128)
    b2 = np.zeros((10, 1))
    return W1, b1, W2, b2

def ReLU(Z): return np.maximum(Z, 0)
def ReLU_deriv(Z): return Z > 0
def softmax(Z):
    Z_shift = Z - np.max(Z, axis=0)
    return np.exp(Z_shift) / np.sum(np.exp(Z_shift), axis=0)

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)     # LỚP NÀY CHÍNH LÀ VECTOR ĐẶC TRƯNG! (128 units)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, 10))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 -= alpha * dW1; b1 -= alpha * db1    
    W2 -= alpha * dW2; b2 -= alpha * db2    
    return W1, b1, W2, b2

def get_predictions(A2): return np.argmax(A2, 0)
def get_accuracy(predictions, Y): return np.sum(predictions == Y) / Y.size

# --- BẮT ĐẦU HUẤN LUYỆN MẠNG NƠ-RON ---
print("Đang huấn luyện mạng Neural Network để lấy khả năng trích xuất đặc trưng...")
W1, b1, W2, b2 = init_params()
alpha = 0.5
iterations = 500 # Bạn có thể tăng lên 1000-2000 để mạng học kỹ hơn

for i in range(iterations):
    Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X_train)
    dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X_train, Y_train)
    W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
    if i % 100 == 0 or i == iterations - 1:
        print(f"NN Vòng lặp {i:4d} | Độ chính xác Train: {get_accuracy(get_predictions(A2), Y_train)*100:.2f}%")

### BƯỚC 3: TRÍCH XUẤT ĐẶC TRƯNG
Lấy output từ lớp ẩn `A1` của mạng Nơ-ron (bỏ qua lớp softmax phân loại chữ số của Mạng nơ-ron).

In [ ]:
print("Đang trích xuất đặc trưng (Feature Extraction)...")
_, train_features, _, _ = forward_prop(W1, b1, W2, b2, X_train)
_, val_features, _, _   = forward_prop(W1, b1, W2, b2, X_val)
_, test_features, _, _  = forward_prop(W1, b1, W2, b2, X_test_final)

# Chuyển vị ma trận (từ 128 x m thành m x 128) để chuẩn bị dạng bảng đưa vào SVM
train_features = train_features.T
val_features   = val_features.T
test_features  = test_features.T

print(f"Kích thước Vector Đặc Trưng Train: {train_features.shape}")

### BƯỚC 4: HUẤN LUYỆN SUPPORT VECTOR MACHINE (TỰ CODE)

In [ ]:
class LinearSVM_Binary:
    def __init__(self, learning_rate=0.01, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0
        for _ in range(self.n_iters):
            margin = y * (np.dot(X, self.w) - self.b)
            misclassified = margin < 1 # Hàm Hinge Loss 
            X_mis = X[misclassified]
            y_mis = y[misclassified]
            dw = 2 * self.lambda_param * self.w - np.dot(y_mis, X_mis) / n_samples
            db = np.sum(y_mis) / n_samples
            self.w -= self.lr * dw
            self.b -= self.lr * db
            
    def get_score(self, X): 
        return np.dot(X, self.w) - self.b

class MultiClassSVM_OVR:
    def __init__(self, n_classes=10, learning_rate=0.1, lambda_param=0.001, n_iters=1000):
        self.n_classes = n_classes
        self.models = [LinearSVM_Binary(learning_rate, lambda_param, n_iters) for _ in range(n_classes)]
        
    def fit(self, X, y):
        for c in range(self.n_classes):
            y_binary = np.where(y == c, 1, -1)
            self.models[c].fit(X, y_binary)
            
    def predict(self, X):
        scores = np.zeros((X.shape[0], self.n_classes))
        for c in range(self.n_classes):
            scores[:, c] = self.models[c].get_score(X)
        return np.argmax(scores, axis=1)

print("Đang huấn luyện SVM Đa Lớp tự code trên vector đặc trưng...")
custom_svm = MultiClassSVM_OVR(n_classes=10, learning_rate=0.5, lambda_param=0.001, n_iters=2000)
custom_svm.fit(train_features, Y_train)
print("Huấn luyện SVM xong!")

### BƯỚC 5: ĐÁNH GIÁ KẾT QUẢ VÀ XUẤT SUBMISSION

In [ ]:
val_predictions = custom_svm.predict(val_features)
acc = np.sum(val_predictions == Y_val) / len(Y_val)
print(f"\n>>> ĐỘ CHÍNH XÁC CỦA HỆ THỐNG MẠNG NƠ-RON + SVM (100% TỰ CODE BẰNG NUMPY): {acc * 100:.2f}% <<<")

test_predictions = custom_svm.predict(test_features)
submission_df = pd.DataFrame({'ImageId': range(1, len(test_predictions) + 1), 'Label': test_predictions})
submission_df.to_csv('submission_All_In_One_Tu_Dau_Numpy.csv', index=False)
print("Đã lưu kết quả thành công vào submission_All_In_One_Tu_Dau_Numpy.csv!")